In [6]:
import os
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
# Mise à jour de l'import pour inclure root_mean_squared_error
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor, plot_importance
import time


# Loading dataset
df = pd.read_csv("https://get-around-project-jc.s3.us-east-1.amazonaws.com/get_around_pricing_project.csv", index_col=0)

# Dropping rows with anomaly
df = df[(df['mileage'] >= 0) & (df['engine_power'] > 0)]

# Splitting dataset into X features and Target variable
target = 'rental_price_per_day'
Y = df[target]
X = df.drop(target, axis = 1)

# categorizing features
numeric_features = []
categorical_features = []
for i,t in X.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)

print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

# Split our training set and our test set 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Features preprocessing
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) 
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head()) 
X_test = preprocessor.transform(X_test) 
print('...Done.')
print(X_test[0:5,:])

# Set your variables for your environment
EXPERIMENT_NAME="getaround-mlflow-experiment"

# Set tracking URI to your Heroku application
os.environ["APP_URI"]="https://atomik31-mlflow.hf.space"
mlflow.set_tracking_uri(os.environ["APP_URI"])

# Set experiment's info 
mlflow.set_experiment(EXPERIMENT_NAME)

# Get our experiment info
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# Time execution
start_time = time.time()

# Call mlflow autolog
mlflow.sklearn.autolog()


print("Linear Regression Training ...")
run_name = 'linear_regression'
with mlflow.start_run(run_name=run_name) as run:
    model_lr = LinearRegression()
    model_lr.fit(X_train, Y_train)
    print("Training done.")
    Y_train_pred = model_lr.predict(X_train)
    Y_test_pred = model_lr.predict(X_test)
    mlflow.log_metric("training_r2_score",r2_score(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_absolute_error",mean_absolute_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_squared_error",mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_root_mean_squared_error",root_mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("testing_r2_score",r2_score(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_absolute_error",mean_absolute_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_squared_error",mean_squared_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_root_mean_squared_error",root_mean_squared_error(Y_test, Y_test_pred))
    mlflow.end_run()

print("Random Forest Training ...")
run_name = 'random_forest'
with mlflow.start_run(run_name=run_name) as run:
    model_rf = RandomForestRegressor(max_depth=10)
    model_rf.fit(X_train, Y_train)
    
    print("Training done.")
    Y_train_pred = model_rf.predict(X_train)
    Y_test_pred = model_rf.predict(X_test)
    mlflow.log_metric("training_r2_score",r2_score(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_absolute_error",mean_absolute_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_squared_error",mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_root_mean_squared_error",root_mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("testing_r2_score",r2_score(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_absolute_error",mean_absolute_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_squared_error",mean_squared_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_root_mean_squared_error",root_mean_squared_error(Y_test, Y_test_pred))
    mlflow.end_run()

print("Ridge Training ...")
run_name = 'ridge'
with mlflow.start_run(run_name=run_name) as run:
    model = Ridge(alpha=1)
    model.fit(X_train, Y_train)
    print("Training done.")
    Y_train_pred = model.predict(X_train)
    Y_test_pred = model.predict(X_test)
    mlflow.log_metric("training_r2_score",r2_score(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_absolute_error",mean_absolute_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_squared_error",mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_root_mean_squared_error",root_mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("testing_r2_score",r2_score(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_absolute_error",mean_absolute_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_squared_error",mean_squared_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_root_mean_squared_error",root_mean_squared_error(Y_test, Y_test_pred))
   
    mlflow.end_run()

print("Lasso Training ...")
run_name = 'lasso'
with mlflow.start_run(run_name=run_name) as run:
    model_lasso =  Lasso(alpha=1)
    model_lasso.fit(X_train, Y_train)
    print("Training done.")
    Y_train_pred = model_lasso.predict(X_train)
    Y_test_pred = model_lasso.predict(X_test)
    mlflow.log_metric("training_r2_score",r2_score(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_absolute_error",mean_absolute_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_squared_error",mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_root_mean_squared_error",root_mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("testing_r2_score",r2_score(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_absolute_error",mean_absolute_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_squared_error",mean_squared_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_root_mean_squared_error",root_mean_squared_error(Y_test, Y_test_pred))
    
    mlflow.end_run()

print("GridSearchCV RandomForest Training ...")
# Grid of values to be tested
run_name = 'random_forest_gridsearch'
with mlflow.start_run(run_name=run_name) as run:
    params_rf = {
    'max_depth': [16, 18, 20],
    'min_samples_split': [2, 4, 6],
    'n_estimators': [150, 200, 250]
    }
    rf = RandomForestRegressor()
    model_gridrf = GridSearchCV(rf, params_rf, cv=5, verbose=True, n_jobs=-1)
    model_gridrf.fit(X_train, Y_train)
    print("Training done.")
    Y_train_pred = model_gridrf.predict(X_train)
    Y_test_pred = model_gridrf.predict(X_test)
    mlflow.log_metric("training_r2_score",r2_score(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_absolute_error",mean_absolute_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_squared_error",mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_root_mean_squared_error",root_mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("testing_r2_score",r2_score(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_absolute_error",mean_absolute_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_squared_error",mean_squared_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_root_mean_squared_error",root_mean_squared_error(Y_test, Y_test_pred))
    mlflow.log_param("best_params", str(model_gridrf.best_params_)) # Converti en string pour mlflow
    
    mlflow.end_run()

print("XGBRegressor Training ...")
run_name = 'xgbr'
with mlflow.start_run(run_name=run_name) as run:
    model_xgb = XGBRegressor(n_estimators=200, max_depth=7, eta=0.1, subsample=0.7, colsample_bytree=0.8, alpha=0.1, random_state=42)
    model_xgb.fit(X_train, Y_train)
    
    print("Training done.")
    Y_train_pred = model_xgb.predict(X_train)
    Y_test_pred = model_xgb.predict(X_test)
    mlflow.log_metric("training_r2_score",r2_score(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_absolute_error",mean_absolute_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_mean_squared_error",mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("training_root_mean_squared_error",root_mean_squared_error(Y_train, Y_train_pred))
    mlflow.log_metric("testing_r2_score",r2_score(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_absolute_error",mean_absolute_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_mean_squared_error",mean_squared_error(Y_test, Y_test_pred))
    mlflow.log_metric("testing_root_mean_squared_error",root_mean_squared_error(Y_test, Y_test_pred))
   
    mlflow.end_run()

print("All training is done!")
print(f"---Total training time: {time.time()-start_time}")

Found numeric features  ['mileage', 'engine_power']
Found categorical features  ['model_key', 'fuel', 'paint_color', 'car_type', 'private_parking_available', 'has_gps', 'has_air_conditioning', 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']
Performing preprocessings on train set...
     model_key  mileage  engine_power    fuel paint_color car_type  \
432    Citroën   234365           135  diesel       black   estate   
3428   Peugeot   120366           100  diesel       black    sedan   
289    Peugeot   181297           105  diesel       brown   estate   
3280   Citroën   171798           120  diesel       black    sedan   
4118       BMW   186120           135  diesel       black      suv   

      private_parking_available  has_gps  has_air_conditioning  automatic_car  \
432                        True     True                 False          False   
3428                      False     True                 False          False   
289                 

2026/04/13 16:36:47 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/04/13 16:36:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/13 16:37:00 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: Unable to locate credentials


Training done.
🏃 View run linear_regression at: https://atomik31-mlflow.hf.space/#/experiments/6/runs/8dcde34d2d4844cbbfae51dc79206f08
🧪 View experiment at: https://atomik31-mlflow.hf.space/#/experiments/6
Random Forest Training ...


2026/04/13 16:37:13 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/04/13 16:37:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/13 16:37:23 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: Unable to locate credentials


Training done.
🏃 View run random_forest at: https://atomik31-mlflow.hf.space/#/experiments/6/runs/941775aa87f041358cd908d57ed89238
🧪 View experiment at: https://atomik31-mlflow.hf.space/#/experiments/6
Ridge Training ...


...

Training done.
🏃 View run lasso at: https://atomik31-mlflow.hf.space/#/experiments/6/runs/93194411a8cb4c05920a8e5ee5053a90
🧪 View experiment at: https://atomik31-mlflow.hf.space/#/experiments/6
GridSearchCV RandomForest Training ...


2026/04/13 16:38:15 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


Fitting 5 folds for each of 27 candidates, totalling 135 fits


2026/04/13 16:39:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/13 16:39:22 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: Unable to locate credentials


Training done.
🏃 View run random_forest_gridsearch at: https://atomik31-mlflow.hf.space/#/experiments/6/runs/833aa41689544ab1b1a774c6dd02c1f2
🧪 View experiment at: https://atomik31-mlflow.hf.space/#/experiments/6
XGBRegressor Training ...
Training done.
🏃 View run xgbr at: https://atomik31-mlflow.hf.space/#/experiments/6/runs/ce696e88e8ec48be82cebc0ad1de63ec
🧪 View experiment at: https://atomik31-mlflow.hf.space/#/experiments/6
All training is done!
---Total training time: 178.1930708885193
